# Análise Exploratória — Loan Approval Prediction Dataset

Dataset: [Loan Approval Prediction Dataset (Kaggle)](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset)

Este notebook realiza uma análise exploratória de dados (EDA) sobre o dataset de aprovação de empréstimos, com foco em entender o perfil dos solicitantes, os fatores associados à aprovação/rejeição (`loan_status`) e a relação entre score de crédito, renda e ativos.

## Dicionário de Dados

| Coluna | Descrição |
|---|---|
| `loan_id` | Identificador do empréstimo |
| `no_of_dependents` | Número de dependentes do solicitante |
| `education` | Escolaridade do solicitante |
| `self_employed` | Se o solicitante é autônomo |
| `income_annum` | Renda anual |
| `loan_amount` | Valor do empréstimo solicitado |
| `loan_term` | Prazo do empréstimo (em anos) |
| `cibil_score` | Score de crédito (CIBIL) |
| `residential_assets_value` | Valor de ativos residenciais |
| `commercial_assets_value` | Valor de ativos comerciais |
| `luxury_assets_value` | Valor de ativos de luxo |
| `bank_asset_value` | Valor de ativos bancários |
| `loan_status` | Status do empréstimo (Approved / Rejected) |

> **Como obter os dados:** baixe o arquivo `loan_approval_dataset.csv` diretamente do Kaggle (link acima) e coloque-o na mesma pasta deste notebook, ou ajuste a variável `DATA_PATH` abaixo.
>
> **Nota:** os nomes reais das colunas no CSV do Kaggle costumam vir com espaços à frente (ex.: `" income_annum"`). O notebook já trata isso na etapa de carregamento/limpeza de nomes de colunas.


In [ ]:
# Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)


In [ ]:
# Caminho do arquivo de dados
DATA_PATH = "loan_approval_dataset.csv"  # ajuste o caminho se necessário

df = pd.read_csv(DATA_PATH)

# Normalização de nomes de colunas (remove espaços extras, comum neste dataset)
df.columns = df.columns.str.strip()

print(df.shape)
df.head()


## 1. Visão Geral dos Dados

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# Padronizar valores de texto (remover espaços extras nas categorias)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df[["education", "self_employed", "loan_status"]].apply(lambda x: x.unique())


In [ ]:
# Checagem de valores ausentes
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")


In [ ]:
# Checagem de duplicatas e da coluna identificadora
print(f"Linhas duplicadas: {df.duplicated().sum()}")
print(f"loan_id únicos: {df['loan_id'].nunique()} de {len(df)}")


## 2. Engenharia de Variáveis Auxiliares

In [ ]:
df_clean = df.copy()

# Soma total de ativos
asset_cols = ["residential_assets_value", "commercial_assets_value",
              "luxury_assets_value", "bank_asset_value"]
df_clean["total_assets_value"] = df_clean[asset_cols].sum(axis=1)

# Razão empréstimo / renda anual
df_clean["loan_to_income"] = df_clean["loan_amount"] / df_clean["income_annum"]

# Razão empréstimo / total de ativos
df_clean["loan_to_assets"] = df_clean["loan_amount"] / df_clean["total_assets_value"].replace(0, np.nan)

df_clean[["total_assets_value", "loan_to_income", "loan_to_assets"]].describe()


## 3. Distribuição da Variável-Alvo (`loan_status`)

In [ ]:
target_counts = df_clean["loan_status"].value_counts(normalize=True) * 100
print(target_counts)

fig, ax = plt.subplots()
sns.countplot(data=df_clean, x="loan_status", palette="Set2", ax=ax, order=df_clean["loan_status"].value_counts().index)
ax.set_title("Distribuição da Variável-Alvo: Loan Status")
ax.set_xlabel("")
ax.set_ylabel("Quantidade")
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()


**Observação:** diferente de datasets de default de crédito, este costuma ser relativamente mais balanceado entre aprovado/rejeitado (algo próximo de 60%/40%), o que facilita a modelagem posterior sem grandes ajustes de balanceamento.

## 4. Análise Univariada — Variáveis Numéricas

In [ ]:
num_cols = ["no_of_dependents", "income_annum", "loan_amount", "loan_term",
            "cibil_score", "residential_assets_value", "commercial_assets_value",
            "luxury_assets_value", "bank_asset_value"]

fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df_clean[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribuição: {col}")
if len(num_cols) < len(axes):
    for j in range(len(num_cols), len(axes)):
        axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=df_clean[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(f"Boxplot: {col}")
if len(num_cols) < len(axes):
    for j in range(len(num_cols), len(axes)):
        axes[j].axis("off")
plt.tight_layout()
plt.show()


## 5. Análise Univariada — Variáveis Categóricas

In [ ]:
cat_cols = ["education", "self_employed", "no_of_dependents"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(cat_cols):
    order = df_clean[col].value_counts().index
    sns.countplot(data=df_clean, x=col, order=order, palette="viridis", ax=axes[i])
    axes[i].set_title(f"Distribuição: {col}")
    axes[i].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 6. Análise Bivariada — Relação com a Aprovação (`loan_status`)

In [ ]:
# Taxa de aprovação por categoria
approved_label = [c for c in df_clean["loan_status"].unique() if "approv" in c.lower()][0]

for col in ["education", "self_employed"]:
    rate = df_clean.groupby(col)["loan_status"].apply(
        lambda x: (x == approved_label).mean() * 100
    ).sort_values(ascending=False)
    print(f"\nTaxa de aprovação por {col} (%):")
    print(rate.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(["education", "self_employed"]):
    rate = df_clean.groupby(col)["loan_status"].apply(
        lambda x: (x == approved_label).mean() * 100
    ).sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, palette="rocket", ax=axes[i])
    axes[i].set_title(f"Taxa de Aprovação (%) por {col}")
    axes[i].set_ylabel("Taxa de Aprovação (%)")
plt.tight_layout()
plt.show()


**Observação:** normalmente, `education` e `self_employed` isoladamente têm pouca influência na aprovação — o principal fator costuma ser o `cibil_score` (score de crédito).

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 22))
axes = axes.flatten()
plot_cols = num_cols + ["loan_to_income", "total_assets_value"]
df_plot = df_clean.copy()
for i, col in enumerate(plot_cols):
    sns.boxplot(data=df_plot, x="loan_status", y=col, palette="Set2", ax=axes[i])
    axes[i].set_title(f"{col} vs Loan Status")
for j in range(len(plot_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


**Insight central:** o `cibil_score` costuma ser, de longe, o fator mais determinante para aprovação — solicitações rejeitadas tipicamente concentram-se em scores baixos (abaixo de ~550), independentemente de renda ou ativos.

In [ ]:
# Distribuição do CIBIL score por status, com linha de corte visual
plt.figure(figsize=(9, 6))
sns.histplot(data=df_clean, x="cibil_score", hue="loan_status", kde=True,
             palette={approved_label: "steelblue"}, element="step", stat="density", common_norm=False)
plt.title("Distribuição do CIBIL Score por Status do Empréstimo")
plt.xlabel("CIBIL Score")
plt.show()


## 7. Correlação entre Variáveis Numéricas

In [ ]:
df_corr = df_clean.copy()
df_corr["loan_status_num"] = (df_corr["loan_status"] == approved_label).astype(int)

corr = df_corr[num_cols + ["total_assets_value", "loan_to_income", "loan_status_num"]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Matriz de Correlação")
plt.show()


## 8. Análise Cruzada: Renda x Valor do Empréstimo x Aprovação

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_clean.sample(min(3000, len(df_clean)), random_state=42),
    x="income_annum", y="loan_amount", hue="loan_status",
    palette="Set1", alpha=0.6
)
plt.title("Renda Anual vs Valor do Empréstimo (colorido por Status)")
plt.xlabel("Renda Anual")
plt.ylabel("Valor do Empréstimo")
plt.show()


In [ ]:
# Relação entre número de dependentes e taxa de aprovação
dep_rate = df_clean.groupby("no_of_dependents")["loan_status"].apply(
    lambda x: (x == approved_label).mean() * 100
)

plt.figure(figsize=(8, 5))
sns.barplot(x=dep_rate.index, y=dep_rate.values, palette="mako")
plt.title("Taxa de Aprovação por Número de Dependentes")
plt.ylabel("Taxa de Aprovação (%)")
plt.xlabel("Número de Dependentes")
plt.show()


In [ ]:
# Relação entre total de ativos e aprovação, segmentado por faixa de CIBIL score
bins = [0, 500, 650, 750, 900]
labels = ["<500", "500-650", "650-750", "750+"]
df_clean["cibil_band"] = pd.cut(df_clean["cibil_score"], bins=bins, labels=labels)

approval_by_band = df_clean.groupby("cibil_band")["loan_status"].apply(
    lambda x: (x == approved_label).mean() * 100
)

plt.figure(figsize=(8, 5))
sns.barplot(x=approval_by_band.index, y=approval_by_band.values, palette="crest")
plt.title("Taxa de Aprovação por Faixa de CIBIL Score")
plt.ylabel("Taxa de Aprovação (%)")
plt.xlabel("Faixa de CIBIL Score")
plt.show()


## 9. Principais Conclusões

- O `cibil_score` é o preditor mais forte de aprovação/rejeição do empréstimo, muito mais relevante do que renda, ativos ou escolaridade isoladamente.
- Variáveis de ativos (`residential_assets_value`, `commercial_assets_value`, `luxury_assets_value`, `bank_asset_value`) tendem a ter correlação relativamente baixa com a aprovação quando analisadas isoladamente, mas podem ganhar relevância combinadas em razões como `loan_to_assets`.
- `education` e `self_employed` aparentam ter impacto marginal na decisão final.
- Recomenda-se, para modelagem futura: usar `cibil_score` como variável-chave, testar interações (ex.: `cibil_score` x `loan_to_income`), aplicar encoding em variáveis categóricas e comparar modelos como Regressão Logística, Árvores de Decisão e Gradient Boosting.
